# BERT: Bidirectional Encoder Representations from Transformers
## Week 7 — Day 3 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Tokenize text with BERT and understand special tokens ([CLS], [SEP])
- Use a pretrained BERT pipeline for sentiment analysis
- Build a custom sentiment analyzer with full control over tokenizer, model, and post-processing
- Build a Named Entity Recognizer (NER) with BERT
- Compare BERT vs GPT architectures, purposes, and use cases
- Understand BERT's role in Retrieval-Augmented Generation (RAG) systems

**Exercises:**
1. Tokenization with BERT
2. Sentiment Analysis with BERT Pipeline
3. Custom Sentiment Analyzer
4. BERT for Named Entity Recognition
5. Comparing BERT and GPT
6. BERT in RAG Systems

> **Run on Google Colab** — GPU optional.

In [ ]:
!pip install --quiet transformers torch

In [ ]:
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    pipeline,
    BertTokenizer,
)
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} ✓")
print(f"Device  : {device}")

## 🌟 Exercise 1 — Tokenization with BERT

### How BERT Tokenizes Text

BERT uses **WordPiece tokenization**, a sub-word algorithm that splits rare or unknown words into smaller pieces. Special tokens frame every input:

- **`[CLS]`** (token ID 101) — prepended to every sequence. Its final hidden state is used as a pooled representation for classification tasks.
- **`[SEP]`** (token ID 102) — appended after each sentence (or between two sentences in NSP).
- **`[PAD]`** (token ID 0) — used to fill shorter sequences to a uniform batch length.
- **`[MASK]`** (token ID 103) — replaces tokens during Masked Language Model (MLM) pretraining.

WordPiece prefixes sub-word continuation pieces with `##`, e.g. `"tokenization"` → `["token", "##ization"]`.

In [ ]:
# 1. Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(f"Tokenizer loaded ✓  |  vocab size: {tokenizer.vocab_size:,}")

# 2. Sample sentence
sentence = "BERT tokenizes text into subword units for natural language understanding."

# 3. Raw tokenization (without special tokens)
tokens    = tokenizer.tokenize(sentence)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"\n{'─'*60}")
print(f"Original text : {sentence}")
print(f"\nTokens ({len(tokens)}):")
print(f"  {tokens}")
print(f"\nToken IDs:")
print(f"  {token_ids}")

# 4. WordPiece sub-word pieces
subword_pieces = [t for t in tokens if t.startswith("##")]
print(f"\nSub-word continuation pieces (## prefix): {subword_pieces if subword_pieces else 'none'}")

# 5. Encoded with special tokens, padding, truncation
encoded = tokenizer(
    sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=20,
    return_tensors="pt",
)

full_ids   = encoded["input_ids"][0].tolist()
full_toks  = tokenizer.convert_ids_to_tokens(full_ids)
attn_mask  = encoded["attention_mask"][0].tolist()

print(f"\n{'─'*60}")
print(f"Encoded (max_length=20, with special tokens + padding):")
print(f"\n  {'Token':<15} {'ID':>6}  {'Attention':>10}  {'Note'}")
print(f"  {'─'*50}")
for tok, tid, am in zip(full_toks, full_ids, attn_mask):
    note = ""
    if tok == "[CLS]":  note = "← classification token (start)"
    elif tok == "[SEP]": note = "← sentence separator (end)"
    elif tok == "[PAD]": note = "← padding token"
    elif tok.startswith("##"): note = "← sub-word continuation"
    print(f"  {tok:<15} {tid:>6}  {'real' if am == 1 else 'pad':>10}  {note}")

print(f"\nSpecial token IDs  [CLS]={tokenizer.cls_token_id}  "
      f"[SEP]={tokenizer.sep_token_id}  [PAD]={tokenizer.pad_token_id}  "
      f"[MASK]={tokenizer.mask_token_id}")

## 🌟 Exercise 2 — Sentiment Analysis with BERT Pipeline

In [ ]:
# Create a sentiment analysis pipeline using DistilBERT fine-tuned on SST-2
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,
)
print("Pipeline loaded ✓  (distilbert-base-uncased-finetuned-sst-2-english)")

# Test sentences
sentences = [
    "I absolutely loved this movie — the acting was superb!",
    "The service was terrible and the food was cold.",
    "It was an okay experience, nothing special.",
    "The new product exceeded all my expectations.",
    "I've never been so disappointed in my life.",
]

print(f"\n{'─'*65}")
print(f"{'Sentence':<45} {'Label':<12} {'Score':>8}")
print(f"{'─'*65}")

results = sentiment_pipeline(sentences)
for sent, res in zip(sentences, results):
    short = sent[:43] + "…" if len(sent) > 44 else sent
    print(f"{short:<45} {res['label']:<12} {res['score']:>8.4f}")

print(f"{'─'*65}")

## 🌟 Exercise 3 — Building a Custom Sentiment Analyzer

In [ ]:
import re

class BERTSentimentAnalyzer:
    """
    Custom sentiment analyzer built directly on top of AutoTokenizer
    and AutoModelForSequenceClassification — no pipeline abstraction.
    """

    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(device).eval()
        self.id2label  = self.model.config.id2label
        print(f"BERTSentimentAnalyzer loaded ✓")
        print(f"  Model      : {model_name}")
        print(f"  Labels     : {self.id2label}")

    def _preprocess(self, text: str) -> dict:
        """Clean text and tokenize into tensors."""
        text = re.sub(r"\s+", " ", text.strip())       # collapse whitespace
        text = re.sub(r"[^\w\s.!?,'\"-]", "", text)    # remove unusual chars
        return self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
        )

    def predict(self, text: str) -> dict:
        """Return label, confidence, and raw probabilities for a single text."""
        inputs = {k: v.to(device) for k, v in self._preprocess(text).items()}
        with torch.no_grad():
            logits = self.model(**inputs).logits          # (1, num_labels)
        probs      = F.softmax(logits, dim=-1)[0]         # (num_labels,)
        pred_idx   = probs.argmax().item()
        return {
            "text"        : text,
            "label"       : self.id2label[pred_idx],
            "confidence"  : probs[pred_idx].item(),
            "probabilities": {self.id2label[i]: probs[i].item()
                              for i in range(len(probs))},
        }

    def analyze_batch(self, texts: list) -> list:
        return [self.predict(t) for t in texts]


# ── Instantiate & test ───────────────────────────────────────────────────────
analyzer = BERTSentimentAnalyzer()

test_texts = [
    "The concert was absolutely breathtaking — best night of my life!",
    "I waited 45 minutes and no one served me. Total waste of time.",
    "The hotel room was clean but a bit small for the price.",
    "Phenomenal performance by the entire cast. Standing ovation!",
    "Honestly, I regret buying this product. It broke after a week.",
]

print(f"\n{'─'*70}")
for result in analyzer.analyze_batch(test_texts):
    short = result['text'][:50] + "…" if len(result['text']) > 51 else result['text']
    print(f"Text       : {short}")
    print(f"Prediction : {result['label']}  (confidence: {result['confidence']:.4f})")
    for label, prob in result['probabilities'].items():
        bar = "█" * int(prob * 30)
        print(f"  {label:<10} {prob:.4f}  {bar}")
    print(f"{'─'*70}")

## 🌟 Exercise 4 — Understanding BERT for Named Entity Recognition (NER)

### B-I-O Tagging Scheme

BERT NER models use the **B-I-O** (Beginning, Inside, Outside) scheme to tag each token:

| Tag | Meaning | Example |
|---|---|---|
| `B-PER` | Beginning of a person entity | `"Barack"` |
| `I-PER` | Continuation of a person entity | `"Obama"` |
| `B-ORG` | Beginning of an organization | `"Google"` |
| `B-LOC` | Beginning of a location | `"Paris"` |
| `O` | Outside any entity | `"the"`, `"and"` |

Because BERT uses sub-word tokenization, one word may produce multiple tokens. NER post-processing must merge `##` continuation tokens back to their root and assign the root token's label to the whole word.

In [ ]:
class BERTNamedEntityRecognizer:
    """
    Token-level NER using a BERT model fine-tuned on the CoNLL-2003 dataset.
    Merges WordPiece ## sub-word tokens and maps predictions to human-readable entities.
    """

    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(device).eval()
        self.id2label  = self.model.config.id2label
        print(f"BERTNamedEntityRecognizer loaded ✓")
        print(f"  Model  : {model_name}")
        print(f"  Labels : {list(self.id2label.values())}")

    def recognize(self, text: str) -> list[dict]:
        """Return a list of recognized entities with their span and type."""
        inputs     = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        input_ids  = inputs["input_ids"].to(device)
        tokens_raw = self.tokenizer.convert_ids_to_tokens(input_ids[0])

        with torch.no_grad():
            logits = self.model(**{k: v.to(device) for k, v in inputs.items()}).logits
        preds = logits.argmax(dim=-1)[0].tolist()   # one label per token

        # Merge ## sub-word pieces back into whole words
        entities, current_word, current_label = [], "", "O"

        for token, pred_id in zip(tokens_raw, preds):
            if token in ("[CLS]", "[SEP]", "[PAD]"):
                continue
            label = self.id2label[pred_id]

            if token.startswith("##"):
                current_word += token[2:]           # strip ## and append
            else:
                # Save previous word if it carried a non-O entity
                if current_word and current_label != "O":
                    entity_type = current_label.split("-")[-1]  # strip B-/I-
                    entities.append({"word": current_word, "entity": entity_type,
                                     "bio": current_label})
                current_word  = token
                current_label = label

        # Flush the last word
        if current_word and current_label != "O":
            entities.append({"word": current_word, "entity": current_label.split("-")[-1],
                             "bio": current_label})

        return entities

    def display(self, text: str):
        print(f"Text: {text}\n")
        entities = self.recognize(text)
        if not entities:
            print("  No named entities found.")
            return
        print(f"  {'Word':<20} {'Entity type':<12} {'BIO tag'}")
        print(f"  {'─'*44}")
        for ent in entities:
            print(f"  {ent['word']:<20} {ent['entity']:<12} {ent['bio']}")
        print()


# ── Instantiate & test ────────────────────────────────────────────────────────
ner = BERTNamedEntityRecognizer()

sample_texts = [
    "Barack Obama was born in Honolulu, Hawaii and served as the 44th President of the United States.",
    "Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cupertino, California.",
    "The Eiffel Tower in Paris attracts millions of tourists every year.",
    "Elon Musk, CEO of Tesla and SpaceX, announced a new partnership with NASA.",
]

print()
for text in sample_texts:
    ner.display(text)
    print("─" * 60)

## 🌟 Exercise 5 — Comparing BERT and GPT

### Architecture & Purpose

| Aspect | BERT | GPT |
|---|---|---|
| **Architecture** | Encoder-only (bidirectional Transformer) | Decoder-only (causal/unidirectional Transformer) |
| **Attention direction** | Attends to **all** tokens (left + right simultaneously) | Attends only to **past** tokens (causal masking) |
| **Pretraining objective** | Masked Language Modeling (MLM) + Next Sentence Prediction (NSP) | Causal Language Modeling (CLM) — predict next token |
| **Primary purpose** | **Language understanding** — encoding rich representations | **Language generation** — producing fluent text sequences |
| **Common use cases** | Sentence classification, NER, QA, semantic search, NLI | Text generation, chatbots, code completion, summarization |
| **Strengths** | Deeply contextualized representations; strong on NLU benchmarks (GLUE, SQuAD) | Excellent at open-ended generation; scales to very large models |
| **Weaknesses** | Cannot generate text naturally (not autoregressive) | Weaker at tasks requiring full bidirectional context |
| **Fine-tuning style** | Add a task head on top of [CLS] or token representations | Prompt-based or add a language-modeling head for generation |
| **Notable variants** | RoBERTa, DistilBERT, ALBERT, ELECTRA, DeBERTa | GPT-2, GPT-3, GPT-4, InstructGPT, ChatGPT |
| **Input representation** | Token + Segment + Positional embeddings | Token + Positional embeddings |
| **Output** | Contextual token embeddings | Next-token probability distribution |

### When to Use Each

**Choose BERT (or variants) when:**
- You need to *understand* and classify text (sentiment, intent, NLI)
- You are extracting information (NER, relation extraction)
- You need dense embeddings for semantic search / retrieval
- You have a well-defined classification or regression target

**Choose GPT (or variants) when:**
- You need to *generate* coherent, long-form text
- You are building a chatbot, assistant, or autocomplete system
- You want few-shot or zero-shot prompting without fine-tuning
- The task is open-ended and the output format is not fixed

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# ── BERT diagram ──────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("BERT — Bidirectional Encoder", fontweight="bold", fontsize=12, pad=10)

input_toks = ["[CLS]", "Paris", "is", "beautiful", "[SEP]"]
colors_b   = ["#e8a838", "#4e91d4", "#4e91d4", "#4e91d4", "#e8a838"]
for i, (tok, col) in enumerate(zip(input_toks, colors_b)):
    x = 1 + i * 1.6
    ax.add_patch(mpatches.FancyBboxPatch((x-0.5, 1), 1.2, 0.7,
                 boxstyle="round,pad=0.05", facecolor=col, alpha=0.85))
    ax.text(x + 0.1, 1.35, tok, ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")

ax.add_patch(mpatches.FancyBboxPatch((0.7, 2.5), 8.2, 3.5,
             boxstyle="round,pad=0.1", facecolor="#d4eaf7", alpha=0.7, linewidth=1.5))
ax.text(4.8, 4.35, "Multi-layer Bidirectional\nTransformer Encoder", ha="center", va="center",
        fontsize=10, fontweight="bold", color="#1a5f8a")
ax.text(4.8, 3.2, "← each token attends to ALL other tokens →", ha="center", va="center",
        fontsize=8, color="#1a5f8a", style="italic")

for i, (tok, col) in enumerate(zip(input_toks, colors_b)):
    x = 1 + i * 1.6
    ax.annotate("", (x + 0.1, 2.5), (x + 0.1, 1.7),
                arrowprops=dict(arrowstyle="->", color="#555", lw=1.2))
    ax.annotate("", (x + 0.1, 7.2), (x + 0.1, 6.0),
                arrowprops=dict(arrowstyle="->", color="#555", lw=1.2))
    ax.add_patch(mpatches.FancyBboxPatch((x-0.5, 7.2), 1.2, 0.7,
                 boxstyle="round,pad=0.05", facecolor="#5cb85c", alpha=0.85))
    ax.text(x + 0.1, 7.55, f"h{i}", ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")

ax.text(4.8, 8.5, "Contextual Embeddings (used for any NLU task)", ha="center", fontsize=9, color="#333")

# ── GPT diagram ───────────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis("off")
ax2.set_title("GPT — Causal Decoder", fontweight="bold", fontsize=12, pad=10)

input_toks2 = ["The", "sky", "is", "blue"]
for i, tok in enumerate(input_toks2):
    x = 1 + i * 1.8
    ax2.add_patch(mpatches.FancyBboxPatch((x-0.5, 1), 1.3, 0.7,
                  boxstyle="round,pad=0.05", facecolor="#9b59b6", alpha=0.85))
    ax2.text(x + 0.15, 1.35, tok, ha="center", va="center", fontsize=9, color="white", fontweight="bold")

ax2.add_patch(mpatches.FancyBboxPatch((0.7, 2.5), 8.2, 3.5,
              boxstyle="round,pad=0.1", facecolor="#f0e6fa", alpha=0.7, linewidth=1.5))
ax2.text(4.8, 4.35, "Multi-layer Causal\nTransformer Decoder", ha="center", va="center",
         fontsize=10, fontweight="bold", color="#6c2e9e")
ax2.text(4.8, 3.2, "← each token attends only to past tokens →", ha="center", va="center",
         fontsize=8, color="#6c2e9e", style="italic")

for i, tok in enumerate(input_toks2):
    x = 1 + i * 1.8
    ax2.annotate("", (x + 0.15, 2.5), (x + 0.15, 1.7),
                 arrowprops=dict(arrowstyle="->", color="#555", lw=1.2))
    ax2.annotate("", (x + 0.15, 7.2), (x + 0.15, 6.0),
                 arrowprops=dict(arrowstyle="->", color="#555", lw=1.2))
    ax2.add_patch(mpatches.FancyBboxPatch((x-0.5, 7.2), 1.3, 0.7,
                  boxstyle="round,pad=0.05", facecolor="#e07b39", alpha=0.85))
    ax2.text(x + 0.15, 7.55, f"P(t{i+2})", ha="center", va="center", fontsize=7.5,
             color="white", fontweight="bold")

ax2.text(4.8, 8.5, "Next-token probabilities → autoregressive generation", ha="center", fontsize=9, color="#333")

plt.suptitle("BERT vs GPT — Architecture Comparison", fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("bert_vs_gpt.png", dpi=120, bbox_inches="tight")
plt.show()
print("Diagram saved ✓")

## 🌟 Exercise 6 — BERT in Retrieval-Augmented Generation (RAG)

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is an architecture that augments a generative language model (e.g. GPT-4) with a retrieval component that fetches relevant external documents at inference time. This allows the model to answer questions about information it was never trained on — such as private databases, real-time news, or company documentation — without needing to retrain.

---

### BERT's Role in the Retrieval Component

BERT (and its variants like RoBERTa, E5, or sentence-transformers) is used as the **bi-encoder retriever**:

1. **Document embedding**: every document (or chunk) in the knowledge base is passed through BERT. The `[CLS]` token's final hidden state (or mean-pooled output) becomes a **dense vector embedding** that encodes the document's semantic meaning.

2. **Query embedding**: when a user submits a query, it is passed through the **same** BERT encoder to produce a dense query vector.

3. **Similarity search**: the query vector is compared against all document vectors using **cosine similarity** (or dot product). The top-k most similar documents are retrieved.

4. **Vector database**: in production, document embeddings are stored in a **vector database** (e.g. FAISS, Pinecone, Weaviate, Chroma) which enables approximate nearest-neighbor search at scale in milliseconds.

---

### Full RAG Pipeline

```
User query
    │
    ▼
[BERT Encoder] ──► Query vector (dense)
    │
    ▼
Vector DB similarity search ──► Top-k relevant document chunks
    │
    ▼
[Prompt construction]
  "Context: {chunk1} {chunk2}…  Question: {user query}  Answer:"
    │
    ▼
[GPT / LLaMA / Mistral] ──► Grounded, factual generated answer
```

---

### Why BERT Rather Than GPT for Retrieval?

| | BERT (Bi-encoder) | GPT |
|---|---|---|
| **Output** | Dense embedding of entire input | Next-token probabilities |
| **Speed** | Fast — embed documents offline, query online | Slow — must process both query and document jointly |
| **Bidirectionality** | ✓ attends left + right — richer semantic representation | ✗ causal only |
| **Use in RAG** | Retriever (encode & match) | Generator (read context and answer) |

---

### Concrete Example

> **Query**: "What is Anthropic's safety approach?"
>
> 1. BERT encodes the query → vector `q`
> 2. FAISS finds the 5 chunks from Anthropic's documentation closest to `q`
> 3. Those chunks + the query are packed into a prompt for GPT-4
> 4. GPT-4 generates a grounded answer, citing the retrieved context
>
> **Result**: the answer reflects *current* documentation, not just GPT-4's training data (which has a knowledge cutoff).

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ── Simulate a minimal RAG retrieval step ────────────────────────────────────
# Use a sentence-transformer-style BERT to embed documents and a query,
# then retrieve the most semantically similar document.

EMBED_MODEL = "bert-base-uncased"
tok  = AutoTokenizer.from_pretrained(EMBED_MODEL)
bert = AutoModel.from_pretrained(EMBED_MODEL).to(device).eval()

def mean_pool(model_output, attention_mask):
    """Mean-pool token embeddings, excluding padding."""
    token_emb = model_output.last_hidden_state              # (B, T, D)
    mask_exp  = attention_mask.unsqueeze(-1).float()        # (B, T, 1)
    return (token_emb * mask_exp).sum(1) / mask_exp.sum(1)  # (B, D)

def embed(texts: list[str]) -> torch.Tensor:
    enc = tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = bert(**enc)
    return F.normalize(mean_pool(out, enc["attention_mask"]), dim=-1)  # unit vectors


# Knowledge base — document chunks
documents = [
    "BERT is a bidirectional encoder pretrained on masked language modeling and next sentence prediction.",
    "GPT is an autoregressive decoder trained to predict the next token, making it ideal for text generation.",
    "RAG systems combine a dense retriever (typically BERT-based) with a generative model like GPT.",
    "Vector databases like FAISS enable millisecond approximate nearest-neighbor search over millions of embeddings.",
    "Fine-tuning adapts pretrained models to downstream tasks using small labeled datasets.",
    "Named Entity Recognition identifies tokens corresponding to people, organizations, and locations in text.",
]

query = "How does BERT help retrieve documents in a RAG system?"

# Embed everything
doc_embs   = embed(documents)   # (N, D)
query_emb  = embed([query])     # (1, D)

# Cosine similarity (vectors are already normalized → dot product = cosine)
scores = (query_emb @ doc_embs.T)[0]   # (N,)
ranked = scores.argsort(descending=True)

print(f"Query: \"{query}\"\n")
print(f"{'─'*72}")
print(f"{'Rank':<6} {'Score':>7}  Document")
print(f"{'─'*72}")
for rank, idx in enumerate(ranked):
    marker = " ◄ top match" if rank == 0 else ""
    print(f"  {rank+1:<4} {scores[idx].item():>7.4f}  {documents[idx]}{marker}")
print(f"{'─'*72}")

print(f"\nTop retrieved context injected into GPT prompt:")
top_doc = documents[ranked[0]]
prompt  = (f"Context: {top_doc}\n\n"
           f"Question: {query}\n\n"
           f"Answer: Based on the context, BERT-based encoders embed both the query and documents "
           f"into dense vectors; cosine similarity identifies the most relevant chunks, which are "
           f"then prepended to the generator's prompt so it can produce a grounded answer.")
print(f"\n{prompt}")

## Summary & Key Takeaways

| Exercise | Concept | Key point |
|---|---|---|
| 1 | BERT Tokenization | WordPiece sub-words + `[CLS]`/`[SEP]` special tokens; `##` marks continuation pieces |
| 2 | Sentiment pipeline | `pipeline()` wraps tokenizer + model + post-processing in one call; returns label + confidence |
| 3 | Custom analyzer | `_preprocess` → `model(**inputs).logits` → `softmax` → `argmax`; full control over each step |
| 4 | NER with B-I-O | Token-level classification; `##` pieces must be merged back to whole words before display |
| 5 | BERT vs GPT | BERT = bidirectional encoder (understand); GPT = causal decoder (generate) |
| 6 | BERT in RAG | BERT encodes query + docs → cosine similarity → top-k chunks → GPT prompt for grounded answers |

**Core insight**: BERT's bidirectionality makes its embeddings richer for *understanding and retrieval*, while GPT's autoregressive design makes it powerful for *open-ended generation*. RAG brings them together — BERT retrieves, GPT generates.